# Stage 2 — Inference Acceleration
## Notebook 3: SGLang Basics

**Objective:** Learn SGLang — an alternative LLM serving framework with RadixAttention for prefix caching, a structured-generation DSL, speculative decoding, and constrained generation (JSON mode, regex).

**Prerequisites:** vLLM familiarity (Notebook 2), basic Python decorator knowledge.

In [ ]:
# ============================
# Cell 1: Install dependencies
# ============================
!pip install -q "sglang[all]" openai
!pip install -q aiohttp numpy matplotlib pydantic

print("Dependencies installed.")
print("Note: SGLang requires CUDA GPU for actual inference.")

## 1. What is SGLang?

SGLang is a fast serving framework for LLMs with two key innovations:

- **RadixAttention**: Automatic KV cache reuse via a radix tree (prefix trie) -- shared prefixes across requests are computed only once
- **Structured Generation DSL** (`@sgl.function`): Pythonic API for defining complex LLM workflows with control flow, parallelism, and constrained decoding

### Architecture

```
+---------------------------+
|  @sgl.function DSL        |  <- Python API for generation workflows
+---------------------------+
|  SGLang Runtime           |  <- Backend (SGLang native, vLLM, OpenAI)
+---------------------------+
|  RadixAttention Scheduler |  <- Prefix tree KV cache management
+---------------------------+
|  GPU Executor             |  <- CUDA kernels + tensor parallelism
+---------------------------+
```

In [ ]:
# ========================================
# Cell 3: Verify SGLang installation
# ========================================

try:
    import sglang as sgl
    print(f"SGLang version: {sgl.__version__}")
    print("\nAvailable backends:")
    print("  - sglang (native)  -- RadixAttention + prefix caching")
    print("  - vllm             -- Use vLLM as backend engine")
    print("  - openai           -- Use any OpenAI-compatible endpoint")
    print("  - anthropic        -- Use Claude models")
    print("  - litellm          -- Multi-provider proxy")
except ImportError:
    print("SGLang not installed. Run: pip install 'sglang[all]'")

# Launch command reference
print("\n" + "=" * 60)
print("SGLang Server Launch (GPU REQUIRED):")
print("=" * 60)
print("""
# CLI launch:
python -m sglang.launch_server \\
    --model-path Qwen/Qwen2.5-0.5B \\
    --host 0.0.0.0 --port 30000 \\
    --tp-size 1 --mem-fraction-static 0.85 \\
    --context-length 8192

# Python API launch:
from sglang.utils import launch_server
server, port = launch_server(model_path="Qwen/Qwen2.5-0.5B", port=30000)
""")

## 2. RadixAttention -- Automatic Prefix Caching

**RadixAttention** is SGLang's key innovation. It uses a **radix tree** (prefix trie) to automatically cache and reuse KV cache for shared prefixes across requests.

This is especially powerful for:
- **Multi-turn conversations**: system prompt + conversation history is shared
- **Few-shot prompts**: shared exemplars
- **Batch processing**: same instruction, different inputs

### How it works

```
Without prefix caching:
  Request 1: [sys| user_a| ...]  -> computes full KV cache
  Request 2: [sys| user_b| ...]  -> recomputes sys prompt KV (REDUNDANT)
  Request 3: [sys| user_c| ...]  -> recomputes sys prompt KV (REDUNDANT)

With RadixAttention:
          [sys]          <- cached after first request
         /  |  \
    user_a user_b user_c  <- only compute unique suffix

  Request 1: computes [sys] + [user_a|...]  -> caches [sys]
  Request 2: reuses [sys], computes [user_b|...]  (saves sys tokens)
  Request 3: reuses [sys], computes [user_c|...]  (saves sys tokens)
```

### Performance impact
- Multi-turn chat: 30-50% TTFT (time-to-first-token) reduction
- Batch inference with shared instructions: 5-10x throughput improvement
- Cost: no extra memory -- Radix tree uses LRU eviction on KV cache

In [ ]:
# ============================================
# Cell 5: RadixAttention simulation (conceptual)
# ============================================

class RadixNode:
    def __init__(self, token_id=None):
        self.token_id = token_id
        self.children = {}       # token_id -> RadixNode
        self.kv_cache = None
        self.ref_count = 0

class RadixCache:
    def __init__(self):
        self.root = RadixNode()
        self.total_cached = 0
        self.total_computed = 0

    def match_prefix(self, tokens):
        node = self.root
        matched = 0
        for token in tokens:
            if token in node.children:
                node = node.children[token]
                matched += 1
            else:
                break
        return matched, node

    def insert(self, tokens):
        node = self.root
        for token in tokens:
            if token not in node.children:
                node.children[token] = RadixNode(token)
            node = node.children[token]
            node.ref_count += 1

# Simulate prefix caching
cache = RadixCache()
sys_prompt = [1, 2, 3, 4, 5]
requests = [
    sys_prompt + [10, 11, 12],
    sys_prompt + [20, 21, 22],
    sys_prompt + [30, 31, 32],
]

for i, req in enumerate(requests):
    matched, _ = cache.match_prefix(req)
    cache.insert(req)
    cache.total_computed += len(req)
    cache.total_cached += matched
    print(f"Request {i+1}: {len(req)} tokens, {matched} cached, {len(req)-matched} computed")

total = sum(len(r) for r in requests)
print(f"\nWith caching: {cache.total_computed} tokens computed")
print(f"Without caching: {total} tokens computed")
print(f"Speedup: {total / cache.total_computed:.1f}x")

## 3. The `@sgl.function` DSL for Structured Generation

SGLang provides a Pythonic DSL for defining generation workflows. The `@sgl.function` decorator turns a Python function into a generation program, where `s` is a state object that accumulates prompts and generated tokens.

In [ ]:
# ==========================================
# Cell 7: @sgl.function -- Basic generation
# ==========================================

import sglang as sgl

@sgl.function
def simple_qa(s, question: str):
    """Simple question-answering function."""
    s += sgl.system("You are a helpful assistant. Answer concisely.")
    s += sgl.user(question)
    s += sgl.assistant(sgl.gen("answer", max_tokens=256))

# Run with backend (requires SGLang server)
# backend = sgl.RuntimeEndpoint("http://localhost:30000")
# state = simple_qa.run(question="What is the capital of Japan?", backend=backend)
# print(state["answer"])

print("Defined: simple_qa(question) -> answer")
print("DSL primitives: sgl.system(), sgl.user(), sgl.assistant(), sgl.gen()")

In [ ]:
# ===============================================
# Cell 8: Advanced DSL -- Parallel/branching generation
# ===============================================

@sgl.function
def parallel_essay(s, topic: str):
    """Generate 3 perspectives from shared prefix."""
    s += sgl.system("You write short essays from different perspectives.")
    s += sgl.user(f"Topic: {topic}")
    s += sgl.assistant("Here are 3 perspectives:\n")
    for i in range(3):
        s += f"Perspective {i+1}: " + sgl.gen(f"p{i}", max_tokens=150, stop="\n\n")
        s += "\n"

print("Defined: parallel_essay(topic) -- 3 perspectives from shared prefix")
print("Branches 2 and 3 benefit from RadixAttention caching branch 1's prefix.")

In [ ]:
# ==========================================
# Cell 9: Constrained (choices) generation
# ==========================================

@sgl.function
def classify_sentiment(s, text: str):
    """Force model to output only valid sentiment labels."""
    s += sgl.system("Classify sentiment as positive, negative, or neutral.")
    s += sgl.user(f"Text: {text}")
    s += sgl.assistant("Sentiment: ")
    s += sgl.gen("sentiment", max_tokens=1,
                 choices=["positive", "negative", "neutral"])

print("Defined: classify_sentiment(text) -- token-level constrained output")
print("The model CANNOT output anything outside [positive, negative, neutral].")

## 4. Multi-Turn Conversation Example

Multi-turn conversations benefit enormously from RadixAttention because each new turn shares the entire conversation prefix.

In [ ]:
# =============================================
# Cell 11: Multi-Turn Conversation with SGLang
# =============================================

@sgl.function
def multi_turn_chat(s, conversation_history: list):
    """Multi-turn conversation with automatic prefix caching."""
    s += sgl.system(
        "You are a customer support agent for TechCorp. "
        "Be professional, empathetic, and solution-oriented. "
        "Products: WidgetPro ($49), WidgetMax ($99), WidgetUltra ($199)."
    )
    for turn in conversation_history:
        s += sgl.user(turn["user"])
        s += sgl.assistant(sgl.gen(f"turn_{turn['id']}", max_tokens=200))

conversation = [
    {"id": "1", "user": "I need help with my WidgetPro order."},
    {"id": "2", "user": "It arrived damaged. What are my options?"},
    {"id": "3", "user": "Can I upgrade to WidgetMax instead?"},
]

print("Multi-turn chat defined.")
print(f"Turns: {len(conversation)}")
print("\nPrefix caching per turn:")
print("  Turn 1: computes sys + user1 + assistant1 (full)")
print("  Turn 2: reuses sys, computes user2 + assistant2")
print("  Turn 3: reuses sys + turn1 + turn2, computes user3 + assistant3")
print("\nTTFT reduction grows with conversation length.")

# To run:
# backend = sgl.RuntimeEndpoint("http://localhost:30000")
# state = multi_turn_chat.run(conversation_history=conversation, backend=backend)

## 5. Speculative Decoding

**Speculative decoding** speeds up generation by using a small draft model to propose tokens, then verifying them with the large target model in parallel. If the target model accepts, you get multiple tokens for the cost of one forward pass.

In [ ]:
# ==============================================
# Cell 13: Speculative Decoding Overview
# ==============================================

print("=" * 60)
print("SPECULATIVE DECODING PIPELINE")
print("=" * 60)
print("""
+-----------------+
| Draft Model     |  (small & fast, e.g., Llama-3-1B)
| Proposes K tokens|
+--------+--------+
         | K draft tokens
         v
+-----------------+
| Target Model    |  (large, e.g., Llama-3-70B)
| Verifies K tokens|  -> ONE forward pass
| in parallel      |  -> Accepts 0-K tokens based on prob match
+-----------------+
         |
         v
  Accepted tokens + continue
""")

print("Speedup: 1.5x - 2.5x (depends on draft/target size ratio)")
print()
print("SGLang launch with speculative decoding:")
print("""
python -m sglang.launch_server \\
    --model-path meta-llama/Llama-3-70B-Instruct \\
    --speculative-algorithm EAGLE \\
    --speculative-draft-model-path meta-llama/Llama-3-8B-Instruct \\
    --speculative-num-steps 5 \\
    --speculative-num-draft-tokens 64
""")
print("\nWhen to use:")
print("  YES: Code gen, summarization (repetitive patterns)")
print("  NO:  High-throughput batching (GPU already saturated)")
print("  NO:  Creative writing (low acceptance rate)")

In [ ]:
# ==============================================
# Cell 14: Speculative decoding speedup model
# ==============================================

import numpy as np
import matplotlib.pyplot as plt

def compute_speedup(acceptance_rate, draft_tokens=5, overhead=0.2):
    expected = 1 + acceptance_rate * draft_tokens
    return expected / (1 + overhead)

rates = np.linspace(0.1, 0.95, 50)
for k, label, color in [(3, 'K=3', 'red'), (5, 'K=5', 'blue'), (8, 'K=8', 'green')]:
    plt.plot(rates, [compute_speedup(ar, k) for ar in rates], linewidth=2, label=label, color=color)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='No speedup')
plt.xlabel('Draft Token Acceptance Rate'); plt.ylabel('Speedup (x)')
plt.title('Speculative Decoding: Speedup vs Acceptance Rate')
plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(0.5, 5)
plt.tight_layout(); plt.show()
print("Code generation often has 0.7-0.85 acceptance -> 2-3x speedup.")

## 6. Constrained Generation (JSON Mode, Regex)

SGLang supports multiple constraint types enforced at the token-sampling level -- the model literally cannot output invalid tokens. This is fundamentally different from post-hoc parsing.

In [ ]:
# ============================================
# Cell 16: JSON generation via regex constraint
# ============================================

@sgl.function
def extract_person(s, text: str):
    """Extract person info with regex-enforced JSON."""
    s += sgl.system("Extract person information. Output ONLY valid JSON.")
    s += sgl.user(f"Text: {text}")
    s += sgl.assistant(
        sgl.gen("person", max_tokens=256,
                regex=r'\{"name": "[^"]+", "age": \d+, "email": "[^"]+@[^"]+\.[^"]+"\}')
    )

print("Defined: extract_person(text) -> guaranteed valid JSON via regex")
print("No try/except needed -- output structurally valid by construction.")

In [ ]:
# ============================================
# Cell 17: Pydantic JSON Schema constraint
# ============================================

print("JSON Schema via Pydantic:")
print("-" * 40)
print("""
from pydantic import BaseModel

class ProductReview(BaseModel):
    product_name: str
    rating: int
    pros: list[str]
    cons: list[str]
    recommended: bool

# Pipeline: Pydantic -> JSON Schema -> Regex -> Token-level constraint
# s += sgl.gen("review", max_tokens=512, json_schema=ProductReview)
""")

print("\nConstraint Types Available:")
print("  1. choices=['a', 'b', 'c']      -- Fixed vocabulary")
print("  2. regex='pattern'              -- Regex pattern enforced per-token")
print("  3. json_schema=PydanticModel    -- Full JSON Schema via Pydantic")
print("  4. stop=['\\n\\n', 'END']        -- Stop sequences")
print("\nAll constraints operate at the TOKEN level, not post-hoc.")

## 7. vLLM vs SGLang -- Comprehensive Comparison

Both are leading open-source LLM serving engines, but with different strengths.

In [ ]:
# ==========================================
# Cell 19: vLLM vs SGLang comparison table
# ==========================================

import pandas as pd

comparison = [
    ["KV cache", "PagedAttention (block-based)", "RadixAttention (radix tree + blocks)"],
    ["Prefix caching", "Automatic (APC), opt-in", "Automatic, always-on via radix tree"],
    ["Structured output", "Guided decoding (outlines, lmfe)", "Built-in: choices, regex, json_schema"],
    ["Generation DSL", "No", "Yes (@sgl.function for complex workflows)"],
    ["OpenAI API", "Full", "Full"],
    ["Speculative decoding", "Limited", "Native: EAGLE, Medusa"],
    ["LoRA adapters", "Yes", "Yes"],
    ["Quantization", "GPTQ, AWQ, FP8, GGUF", "GPTQ, AWQ, FP8"],
    ["Tensor parallelism", "Yes", "Yes"],
    ["Community", "30k+ stars", "10k+ stars (growing)"],
    ["Best for", "High-throughput API serving", "Complex workflows, prefix reuse, structured gen"],
]

df = pd.DataFrame(comparison, columns=["Feature", "vLLM", "SGLang"])
print("vLLM vs SGLang Comparison\n" + "=" * 90)
print(df.to_string(index=False))

print("\n\nDECISION GUIDE:")
print()
print("Choose vLLM when:")
print("  - Drop-in OpenAI API replacement needed")
print("  - Simple single-turn completions at high throughput")
print("  - Largest community and battle-tested codebase")
print()
print("Choose SGLang when:")
print("  - Complex multi-step workflows (agent loops, RAG)")
print("  - Many requests share prefixes (chatbots, few-shot)")
print("  - Structured/constrained generation (JSON, regex)")
print("  - Speculative decoding experimentation")
print("  - Benefit from @sgl.function programming model")

## 8. Benchmark: vLLM vs SGLang

A benchmark script that compares both engines on the same hardware with identical prompts.

In [ ]:
# ==========================================
# Cell 21: Benchmark comparison code
# ==========================================

import asyncio
import aiohttp
import time
from dataclasses import dataclass
from typing import List

@dataclass
class BenchmarkResult:
    engine: str
    total_requests: int
    total_tokens: int
    duration_seconds: float
    tokens_per_second: float
    requests_per_second: float
    p50_latency: float
    p95_latency: float
    p99_latency: float

class LLMBenchmark:
    """Benchmark any OpenAI-compatible endpoint."""
    def __init__(self, base_url: str, model: str):
        self.base_url = base_url.rstrip("/") + "/v1/chat/completions"
        self.model = model

    async def _single_request(self, session, prompt, max_tokens=128):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "temperature": 0.0,
        }
        start = time.perf_counter()
        async with session.post(self.base_url, json=payload) as resp:
            data = await resp.json()
            latency = time.perf_counter() - start
            tokens = data.get("usage", {}).get("completion_tokens", 0)
            return latency, tokens

    async def run(self, prompts: List[str], concurrency=32, max_tokens=128):
        sem = asyncio.Semaphore(concurrency)
        async def bounded(session, p):
            async with sem:
                return await self._single_request(session, p, max_tokens)
        start = time.perf_counter()
        async with aiohttp.ClientSession() as session:
            tasks = [bounded(session, p) for p in prompts]
            raw = await asyncio.gather(*tasks, return_exceptions=True)
        duration = time.perf_counter() - start
        results = [r for r in raw if not isinstance(r, Exception)]
        latencies = sorted([r[0] for r in results])
        total_tokens = sum(r[1] for r in results)
        return BenchmarkResult(
            engine="benchmark", total_requests=len(results),
            total_tokens=total_tokens, duration_seconds=duration,
            tokens_per_second=total_tokens / duration,
            requests_per_second=len(results) / duration,
            p50_latency=latencies[len(latencies)//2],
            p95_latency=latencies[int(len(latencies)*0.95)],
            p99_latency=latencies[int(len(latencies)*0.99)],
        )

# Usage (when both servers are running):
prompts = ["Explain machine learning in simple terms."] * 100
print("Benchmark setup ready.")
print("\nWhen vLLM (port 8000) and SGLang (port 30000) are running:")
print("  vllm_bench = LLMBenchmark('http://localhost:8000', 'Qwen/Qwen2.5-0.5B')")
print("  sglang_bench = LLMBenchmark('http://localhost:30000', 'Qwen/Qwen2.5-0.5B')")
print("  vllm_res = await vllm_bench.run(prompts, concurrency=32)")
print("  sglang_res = await sglang_bench.run(prompts, concurrency=32)")

## 9. Exercise: Build a Structured Extraction Pipeline

Using SGLang's DSL, build a pipeline that takes a paragraph and extracts structured information with guaranteed valid JSON output.

In [ ]:
# ================================================
# Cell 23: Exercise -- Structured extraction pipeline
# ================================================

@sgl.function
def extract_and_classify(s, text: str):
    """
    TODO: Complete this pipeline.

    Steps:
    1. Add system prompt explaining the task
    2. Add user message with the input text
    3. Extract entities using regex-constrained generation
    4. Classify sentiment using choices constraint
    5. Generate one-sentence summary

    Expected output:
    {
        "entities": {"people": [...], "orgs": [...], "locations": [...]},
        "sentiment": "positive" | "negative" | "neutral",
        "summary": "one-sentence summary"
    }
    """
    # TODO: Fill in the implementation
    s += sgl.system(...)  # YOUR CODE
    s += sgl.user(text)
    # Entity extraction with JSON regex
    s += sgl.assistant("Entities: ")
    # s += sgl.gen("entities", max_tokens=..., regex=...)
    # Sentiment classification with choices
    s += sgl.assistant("\nSentiment: ")
    # s += sgl.gen("sentiment", max_tokens=1, choices=[...])
    # Summary
    s += sgl.assistant("\nSummary: ")
    # s += sgl.gen("summary", max_tokens=...)


test_text = (
    "Apple Inc. announced that Tim Cook will visit Tokyo next month "
    "to open a new research lab. The stock market reacted positively."
)
print(f"Test input: {test_text}")
print("\nTODO: Complete extract_and_classify and run against SGLang backend.")

## 10. Key Takeaways

1. **SGLang's RadixAttention** automatically caches shared prefixes via a radix tree, giving 30-90% TTFT reduction for workloads with shared prompts (chatbots, RAG, batch inference).

2. **The `@sgl.function` DSL** provides a Pythonic way to define complex LLM workflows with branching, parallelism, and constrained generation -- all with automatic prefix caching.

3. **Constrained generation** (choices, regex, JSON schema) is enforced at the TOKEN level -- the model cannot output invalid tokens. This is fundamentally different from post-hoc parsing.

4. **Speculative decoding** uses a small draft model to propose tokens and a large target model to verify in parallel, achieving 1.5-2.5x speedup for single-user scenarios.

5. **vLLM vs SGLang**: vLLM excels at simple, high-throughput API serving; SGLang excels at complex workflows, structured generation, and prefix-heavy workloads.

6. **The benchmark class** provided can compare any OpenAI-compatible endpoints, making it easy to evaluate both engines on your hardware.

### What's Next

The next notebook covers **Inference Optimization** -- decoding parameter tuning, load testing with Locust, TTFT/TPOT benchmarking, Flash Attention 2, and continuous batching.